In [ ]:
import requests
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

In [ ]:

model_id = "IDEA-Research/grounding-dino-base" #Hugging face의 grounding-DINO사용
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForZeroShotObjectDetection.from_pretrained(model_id).to(device)
model.eval()

In [ ]:

image_url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(image_url, stream=True).raw).convert("RGB")

text = "a cat. a remote control."

inputs = processor(images=image, text=text, return_tensors="pt").to(device)


In [ ]:

with torch.no_grad():
    outputs = model(**inputs)

results = processor.post_process_grounded_object_detection(
    outputs=outputs,
    input_ids=inputs.input_ids,
    threshold=0.3,            
    text_threshold=0.2,
    target_sizes=[image.size[::-1]]  # (H, W)
)



In [ ]:

fig, ax = plt.subplots(1, figsize=(10, 8))
ax.imshow(image)


for score, box, label in zip(
    results[0]["scores"].cpu(),
    results[0]["boxes"].cpu(),
    results[0]["text_labels"]
):
    x_min, y_min, x_max, y_max = box.tolist()
    width, height = x_max - x_min, y_max - y_min

    rect = patches.Rectangle(
        (x_min, y_min),
        width,
        height,
        linewidth=2,
        edgecolor="red",
        facecolor="none"
    )
    ax.add_patch(rect)

    caption = f"{label} ({score:.2f})"
    ax.text(
        x_min,
        max(0, y_min - 5),
        caption,
        fontsize=12,
        color="white",
        bbox=dict(facecolor="red", alpha=0.5, edgecolor="none", boxstyle="round,pad=0.3")
    )

ax.axis("off")

out_path = "test.png"
plt.tight_layout()
plt.savefig(out_path, dpi=200, bbox_inches="tight")
plt.show()

print(f"Saved: {out_path}")